https://docs.pytorch.org/tutorials/beginner/onnx/export_simple_model_to_onnx_tutorial.html
https://onnxruntime.ai/docs/get-started/with-javascript/web.html


In [6]:
import torch
import torch.nn as nn

In [3]:
class ExampleModel(nn.Module):
    """
    Input:  [batch, 3, H, W]
    Output: [batch, n_outputs]
    """
    def __init__(self, n_outputs=8):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = nn.Linear(16, n_outputs)

        # Initialize fixed weights once so the model never needs training.
        self._init_fixed_weights()

    def _init_fixed_weights(self):
        with torch.no_grad():
            # deterministic, hand-made-ish values
            for name, p in self.named_parameters():
                if "weight" in name:
                    values = torch.linspace(-0.5, 0.5, p.numel(), dtype=p.dtype)
                    p.copy_(values.reshape_as(p))
                elif "bias" in name:
                    p.zero_()

    def forward(self, x):
        x = self.features(x)         # [B, 16, 1, 1]
        x = x.flatten(1)             # [B, 16]
        x = self.head(x)             # [B, n_outputs]
        return x


In [5]:
torch_model = ExampleModel()
torch_model.eval()

# example input
example_inputs = (torch.randn(1, 3, 32, 32),)

torch.onnx.export(
    torch_model,
    example_inputs,
    "example_model.onnx",   # <- required output file
    input_names=["input"],
    output_names=["output"],
    opset_version=17
)

In [ ]:
onnx_program.save("sample_model.onnx")
